# EDOs: problemas de valor inicial

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/12-edo-valor-inicial.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. deduzir os métodos de Euler, de Heun e de Runge-Kutta de ordem 4 e prever a ordem do erro;
> 2. escrever uma EDO de ordem 2, ou um sistema, como um sistema de primeira ordem;
> 3. explicar como um método adaptativo escolhe o passo, e o que `rtol` e `atol` controlam de fato;
> 4. usar `scipy.integrate.solve_ivp` com eventos e saída densa;
> 5. verificar uma simulação longa com quantidades conservadas e com um estudo de tolerância.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import newton
from scipy.special import ellipk
from cnlivro.edo import passo_fixo, rk45

## 1. Problema de engenharia: onde estará o satélite daqui a três dias?

Satélites de comunicação para latitudes altas usam órbitas muito alongadas, do tipo **Molniya**: passam rapidamente perto da Terra no perigeu e ficam horas "pendurados" perto do apogeu, sobre a região que atendem. Considere uma órbita com perigeu a 600 km de altitude e apogeu a 39 700 km. Uma estação em solo precisa apontar a antena, e para isso quer saber **onde estará o satélite daqui a três dias**.

No modelo mais simples, só a gravidade da Terra age sobre o satélite. A segunda lei de Newton dá uma EDO vetorial de **segunda ordem**:

$$
\ddot{\mathbf r} = -\frac{\mu}{\|\mathbf r\|^3}\,\mathbf r, \qquad \mu = 398\,600{,}44 \text{ km}^3/\text{s}^2 ,
$$ (eq-orbita)

com a posição e a velocidade no perigeu como **condições iniciais**. É um **problema de valor inicial**: sabemos o estado agora e queremos o estado depois. Para esse modelo ideal (dois corpos) existe uma solução semianalítica, a equação de Kepler, que usaremos para conferir. Num modelo realista, com o achatamento da Terra, o arrasto e a atração da Lua, não há fórmula: só a integração numérica.

In [ ]:
MU, R_T = 398600.4418, 6378.137                 # km³/s², km
r_p, r_a = R_T + 600.0, R_T + 39700.0           # perigeu e apogeu (distâncias ao centro), km
a_orb = (r_p + r_a) / 2                         # semieixo maior
e_orb = (r_a - r_p) / (r_a + r_p)               # excentricidade
T_orb = 2 * np.pi * np.sqrt(a_orb**3 / MU)      # período, s
v_p = np.sqrt(MU * (2 / r_p - 1 / a_orb))       # velocidade no perigeu (vis-viva)
u0 = np.array([r_p, 0.0, 0.0, v_p])             # estado inicial: x, y, vx, vy

def orbita(t, u):
    x, y, vx, vy = u
    r3 = (x * x + y * y) ** 1.5
    return np.array([vx, vy, -MU * x / r3, -MU * y / r3])

print(f"excentricidade e = {e_orb:.4f}, período = {T_orb / 3600:.2f} h")
print(f"velocidade no perigeu = {v_p:.3f} km/s, no apogeu = {v_p * r_p / r_a:.3f} km/s")

In [ ]:
ref = solve_ivp(orbita, (0, T_orb), u0, rtol=1e-11, atol=1e-9, dense_output=True)
tt = np.linspace(0, T_orb, 800)
xy = ref.sol(tt)
th = np.linspace(0, 2 * np.pi, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=R_T * np.cos(th) / 1e3, y=R_T * np.sin(th) / 1e3, fill="toself", mode="lines",
                         line=dict(color=CORES["referencia"]), fillcolor="rgba(153,153,153,0.3)", name="Terra"))
fig.add_trace(go.Scatter(x=xy[0] / 1e3, y=xy[1] / 1e3, mode="lines", line=dict(color=CORES["exata"]), name="órbita"))
th_h = np.arange(0, T_orb, 3600.0)
p_h = ref.sol(th_h)
fig.add_trace(go.Scatter(x=p_h[0] / 1e3, y=p_h[1] / 1e3, mode="markers", name="posição a cada hora",
                         marker=dict(color=CORES["destaque"], size=8)))
fig.update_xaxes(title="x (mil km)", scaleanchor="y")
fig.update_yaxes(title="y (mil km)")
fig.update_layout(width=760, height=480, title="Uma volta da órbita: o satélite passa depressa pelo perigeu",
                  legend=dict(orientation="h", y=-0.2))
mostrar(fig)

**O que notar:** os pontos, marcados a cada hora, ficam espremidos perto do apogeu (à esquerda) e muito espaçados no perigeu (à direita), onde o satélite passa a mais de 10 km/s. Uma integração com passo fixo teria de usar, na volta inteira, o passo minúsculo que o perigeu exige. É aqui que um método de **passo adaptativo** se paga.

## 2. A ideia em uma imagem

Uma EDO $y' = f(t, y)$ diz, em cada ponto do plano $(t, y)$, qual é a **inclinação** da solução que passa por ali. Os traços cinzas da figura são esse **campo de direções**. Resolver a EDO é seguir as setas a partir do ponto inicial.

O método de Euler segue a inclinação do ponto onde está, durante um passo inteiro. Ele erra porque a inclinação muda ao longo do passo. O Runge-Kutta de ordem 4 (RK4) "sonda" a inclinação em quatro lugares dentro do passo e anda com uma média ponderada delas. Arraste o slider para mudar o passo $h$, para $y' = y\cos t$, $y(0) = 1$, cuja solução é $y = e^{\sin t}$.

In [ ]:
f_cos = lambda t, y: y * np.cos(t)
exata_cos = lambda t: np.exp(np.sin(t))
T2 = 8.0
# campo de direções: segmentos curtos, num só traço (None separa os segmentos)
tg, yg = np.meshgrid(np.linspace(0.2, T2 - 0.2, 26), np.linspace(0.2, 3.0, 13))
dt_s = 0.12
inc = f_cos(tg, yg)
norma = np.sqrt(1 + inc**2)
xs_c, ys_c = [], []
for t_, y_, m_, n_ in zip(tg.ravel(), yg.ravel(), inc.ravel(), norma.ravel()):
    xs_c += [t_ - dt_s / n_, t_ + dt_s / n_, None]
    ys_c += [y_ - dt_s * m_ / n_, y_ + dt_s * m_ / n_, None]
tt = np.linspace(0, T2, 400)

def quadro_campo(h):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xs_c, y=ys_c, mode="lines", line=dict(color="rgba(153,153,153,0.7)", width=1.2),
                             name="campo de direções", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=tt, y=exata_cos(tt), mode="lines", line=dict(color=CORES["exata"], width=2.5),
                             name="solução exata"))
    for metodo, nome, cor in [("euler", "Euler", CORES["erro"]), ("rk4", "RK4", CORES["aproximacao"])]:
        r = passo_fixo(f_cos, (0, T2), 1.0, h, metodo)
        fig.add_trace(go.Scatter(x=r.t, y=r.y[:, 0], mode="lines+markers", name=nome,
                                 line=dict(color=cor), marker=dict(size=7)))
    e_eu = passo_fixo(f_cos, (0, T2), 1.0, h, "euler").y[-1, 0] - exata_cos(T2)
    e_rk = passo_fixo(f_cos, (0, T2), 1.0, h, "rk4").y[-1, 0] - exata_cos(T2)
    fig.update_xaxes(title="t", range=[0, T2])
    fig.update_yaxes(title="y", range=[0, 3.2])
    fig.update_layout(width=860, height=460, legend=dict(orientation="h", y=-0.2),
                      title=f"h = {numero(h)}: erro em t = 8 de {numero(e_eu, 3)} (Euler) e {numero(e_rk, 4)} (RK4)")
    return fig

hs_q = [1.0, 0.5, 0.25, 0.125]
quadros = [(numero(h), quadro_campo(h)) for h in hs_q]
mostrar(com_controle(quadros, rotulo="h = ", ativo=0, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com $h = 1$, Euler sai da curva logo no primeiro passo, porque segue a inclinação inicial enquanto a solução já está se curvando, e os erros se acumulam. O RK4, com o mesmo passo, fica quase em cima da solução exata. Dividir $h$ por 2 divide o erro de Euler por cerca de 2, e o do RK4 por cerca de 16: são métodos de ordem 1 e 4. Cada passo do RK4 custa quatro avaliações de $f$, mas o ganho de precisão compensa de longe.

A figura abaixo mostra **um** passo do RK4, com $h = 1$, e as quatro inclinações que ele sonda.

In [ ]:
t0_, y0_, h_ = 0.5, exata_cos(0.5), 1.0
k1 = f_cos(t0_, y0_)
k2 = f_cos(t0_ + h_ / 2, y0_ + h_ / 2 * k1)
k3 = f_cos(t0_ + h_ / 2, y0_ + h_ / 2 * k2)
k4 = f_cos(t0_ + h_, y0_ + h_ * k3)
y1_ = y0_ + h_ / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
pontos = [(t0_, y0_, k1, "k₁ (início)"), (t0_ + h_ / 2, y0_ + h_ / 2 * k1, k2, "k₂ (meio, via k₁)"),
          (t0_ + h_ / 2, y0_ + h_ / 2 * k2, k3, "k₃ (meio, via k₂)"), (t0_ + h_, y0_ + h_ * k3, k4, "k₄ (fim, via k₃)")]
fig = go.Figure()
tt = np.linspace(0.3, 1.7, 200)
fig.add_trace(go.Scatter(x=tt, y=exata_cos(tt), mode="lines", line=dict(color=CORES["exata"], width=2.5),
                         name="solução exata"))
cores_k = [CORES["erro"], CORES["terciaria"], CORES["secundaria"], CORES["destaque"]]
for (tp, yp, kp, nome), cor in zip(pontos, cores_k):
    d = 0.18
    fig.add_trace(go.Scatter(x=[tp - d, tp + d], y=[yp - d * kp, yp + d * kp], mode="lines+markers",
                             line=dict(color=cor, width=4), marker=dict(size=[0, 0]), name=nome))
    fig.add_trace(go.Scatter(x=[tp], y=[yp], mode="markers", marker=dict(color=cor, size=10), showlegend=False))
fig.add_trace(go.Scatter(x=[t0_, t0_ + h_], y=[y0_, y1_], mode="lines+markers", name="passo do RK4: média (k₁ + 2k₂ + 2k₃ + k₄)/6",
                         line=dict(color=CORES["aproximacao"], dash="dash"), marker=dict(size=10, symbol="star")))
fig.update_xaxes(title="t")
fig.update_yaxes(title="y")
fig.update_layout(width=860, height=460, title="Um passo do RK4: quatro inclinações e uma média ponderada",
                  legend=dict(x=0.62, y=0.02))
mostrar(fig)
print(f"y(1,5): exato = {exata_cos(1.5):.5f}, RK4 com um passo = {y1_:.5f}, Euler com um passo = {y0_ + h_ * k1:.5f}")

## 3. Formulação matemática

### 3.1 Euler e o erro local

Integrando $y' = f(t, y)$ de $t_n$ a $t_{n+1} = t_n + h$:

$$
y(t_{n+1}) = y(t_n) + \int_{t_n}^{t_{n+1}} f\big(t, y(t)\big)\,dt .
$$ (eq-edo-integral)

Toda a família de métodos deste capítulo sai de aproximar essa integral, que é uma quadratura como as do capítulo 11, com uma dificuldade nova: o integrando depende da própria solução, que ainda não conhecemos. **Euler** usa o valor da ponta esquerda, $y_{n+1} = y_n + h f(t_n, y_n)$ (a "regra do retângulo"). Pela série de Taylor, o erro cometido em **um** passo, o **erro local**, é $\frac{h^2}{2}y''(\xi)$. Para chegar a um instante fixo são necessários $T/h$ passos, e o **erro global** é de ordem $h$: Euler é de **ordem 1**, como medimos no capítulo 1.

Um método é de **ordem $p$** quando o erro local é $O(h^{p+1})$ e o global, $O(h^p)$.

### 3.2 Métodos de Runge-Kutta

**Heun** (ou Euler melhorado) usa a regra do trapézio na integral {eq}`eq-edo-integral`, com o valor de $y_{n+1}$ do lado direito **previsto** por um passo de Euler:

$$
k_1 = f(t_n, y_n), \quad k_2 = f(t_n + h,\; y_n + h k_1), \quad y_{n+1} = y_n + \frac{h}{2}(k_1 + k_2).
$$

É de ordem 2 e custa duas avaliações de $f$ por passo. A ideia geral dos métodos de **Runge-Kutta** é essa: avaliar $f$ em $s$ **estágios** dentro do passo e combinar as inclinações com pesos escolhidos para que a expansão de Taylor do método coincida com a da solução até a maior potência possível de $h$. O mais famoso é o **RK4**:

$$
\begin{aligned}
k_1 &= f(t_n, y_n), & k_2 &= f\!\left(t_n + \tfrac h2,\; y_n + \tfrac h2 k_1\right),\\
k_3 &= f\!\left(t_n + \tfrac h2,\; y_n + \tfrac h2 k_2\right), & k_4 &= f(t_n + h,\; y_n + h k_3),
\end{aligned}
\qquad
y_{n+1} = y_n + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right).
$$ (eq-rk4)

Se $f$ depende só de $t$, o RK4 é exatamente a regra de Simpson: pesos $1, 4, 1$ em $t_n$, $t_n + h/2$ e $t_n + h$. Ele é de **ordem 4** e custa quatro avaliações por passo. Um método de Runge-Kutta se resume numa tabela de coeficientes (a **tabela de Butcher**); é assim que as bibliotecas descrevem os seus métodos.

Dois cuidados práticos:

- Os estágios usam $t_n + h/2$ e $t_n + h$. Se a EDO tem uma força externa dada só em instantes $t_n$ (uma tabela de medidas), é preciso interpolá-la, ou a ordem cai.
- Para uma EDO **sem solução exata conhecida**, a ordem se mede com um estudo de refinamento, como no capítulo 10: com $h$, $h/2$ e $h/4$, a razão das diferenças sucessivas é $2^p$.

### 3.3 Sistemas e EDOs de ordem mais alta

Os métodos acima valem sem mudança para um **vetor** $\mathbf y$: basta trocar $y$ por $\mathbf y$ e $f$ por $\mathbf f$. Uma EDO de ordem 2 vira um sistema de ordem 1 introduzindo a velocidade como nova incógnita. Para a órbita {eq}`eq-orbita`, com $\mathbf u = (x, y, v_x, v_y)$:

$$
\mathbf u' = \big(v_x,\; v_y,\; -\mu x/r^3,\; -\mu y/r^3\big), \qquad r = \sqrt{x^2 + y^2} .
$$

É o que a função `orbita` da seção 1 faz. Uma EDO de ordem $m$ vira um sistema de $m$ equações de ordem 1, com $m$ condições iniciais.

### 3.4 Passo adaptativo

Um método adaptativo estima o erro local em cada passo e ajusta $h$ para que ele fique abaixo de uma tolerância. A estimativa vem de um **par embutido**: com os mesmos estágios, calculam-se duas soluções, de ordens $p$ e $p - 1$, e a diferença entre elas estima o erro local da menos precisa. O par de **Dormand-Prince 5(4)**, usado por `solve_ivp(method="RK45")` e por `cnlivro.edo.rk45`, tem 7 estágios, dos quais o último é reaproveitado como o primeiro do passo seguinte.

O passo é aceito se o erro estimado, componente a componente, satisfaz

$$
|\text{erro}_i| \le \texttt{atol} + \texttt{rtol}\cdot|y_i| ,
$$ (eq-tolerancia)

e o próximo passo é escolhido para que o erro fique perto do limite: $h_{\text{novo}} = 0{,}9\,h\,(1/\text{erro relativo})^{1/5}$. Se o erro passa do limite, o passo é **rejeitado** e refeito com $h$ menor.

Atenção ao que {eq}`eq-tolerancia` promete: ela limita o erro **local**, de **cada passo**. O erro **global**, no fim da integração, é o acúmulo dos erros locais, propagados pela dinâmica do problema, e pode ser muito maior que `rtol`. Os valores padrão de `solve_ivp` são `rtol=1e-3` e `atol=1e-6`: três algarismos por passo, pensados para um gráfico, não para uma previsão de longo prazo.

### 3.5 `solve_ivp` na prática

`scipy.integrate.solve_ivp(f, (t0, tf), y0, method=..., rtol=..., atol=...)` reúne vários métodos: `RK45` (o padrão), `DOP853` (ordem 8, para tolerâncias apertadas), `RK23` (ordem 3, para tolerâncias frouxas) e os implícitos `Radau`, `BDF` e `LSODA`, para problemas rígidos (capítulo 13). Três recursos que todo pedido a um agente deve mencionar:

- **`dense_output=True`** devolve um interpolante, `sol.sol(t)`, com a mesma ordem do método, para obter a solução em qualquer instante sem refazer a integração.
- **`events`** localiza instantes em que uma função do estado se anula (o satélite cruza um plano, um objeto toca o chão), resolvendo uma equação de raiz (capítulo 3) com o interpolante.
- **`sol.status`** e **`sol.message`** dizem se a integração chegou ao fim. Um resultado com `status = -1` é parcial.

## 4. À mão

Para $y' = y$, $y(0) = 1$, com um passo $h = 0{,}5$ (a solução exata é $y(0{,}5) = e^{0{,}5} = 1{,}648721$):

- **Euler:** $y_1 = 1 + 0{,}5 \times 1 = 1{,}5$. Erro de $0{,}149$.
- **Heun:** $k_1 = 1$, $k_2 = f(0{,}5;\ 1{,}5) = 1{,}5$, $y_1 = 1 + 0{,}25\,(1 + 1{,}5) = 1{,}625$. Erro de $0{,}024$.
- **RK4:** $k_1 = 1$; $k_2 = 1 + 0{,}25 \times 1 = 1{,}25$; $k_3 = 1 + 0{,}25 \times 1{,}25 = 1{,}3125$; $k_4 = 1 + 0{,}5 \times 1{,}3125 = 1{,}65625$. Então $y_1 = 1 + \frac{0{,}5}{6}(1 + 2{,}5 + 2{,}625 + 1{,}65625) = 1{,}6484375$. Erro de $0{,}00028$.

Para $y' = y$, cada método multiplica $y$ por um polinômio em $h$: $1 + h$, $1 + h + h^2/2$ e $1 + h + h^2/2 + h^3/6 + h^4/24$. São os primeiros termos da série de $e^h$, e o número de termos certos é a ordem.

**Uma EDO de ordem 2.** O pêndulo, $\theta'' = -(g/L)\sin\theta$, vira o sistema $\theta' = \omega$, $\omega' = -(g/L)\sin\theta$. Com $g/L = 9{,}81$ s⁻², $\theta_0 = 1$ rad, $\omega_0 = 0$ e um passo de Euler com $h = 0{,}1$ s: $\theta_1 = 1 + 0{,}1 \times 0 = 1$ e $\omega_1 = 0 - 0{,}1 \times 9{,}81 \sin 1 = -0{,}8255$ rad/s.

In [ ]:
for metodo in ["euler", "heun", "rk4"]:
    y1 = passo_fixo(lambda t, y: y, (0, 0.5), 1.0, 0.5, metodo).y[-1, 0]
    print(f"{metodo:6s}: y(0,5) = {y1:.7f}   erro = {np.exp(0.5) - y1:.2e}")
pend = lambda t, u: np.array([u[1], -9.81 * np.sin(u[0])])
print("pêndulo, um passo de Euler:", passo_fixo(pend, (0, 0.1), [1.0, 0.0], 0.1, "euler").y[-1])

## 5. Explore

### 5.1 Precisão em função do custo

O custo de uma integração é o número de avaliações de $f$. Para $y' = y\cos t$ em $[0, 8]$, comparamos os métodos de passo fixo com o RK45 adaptativo, variando o passo ou a tolerância:

In [ ]:
f_cos = lambda t, y: y * np.cos(t)
exato8 = np.exp(np.sin(8.0))
fig = go.Figure()
cores = {"euler": CORES["erro"], "heun": CORES["secundaria"], "rk4": CORES["aproximacao"]}
nomes = {"euler": "Euler (ordem 1)", "heun": "Heun (ordem 2)", "rk4": "RK4 (ordem 4)"}
for metodo in ["euler", "heun", "rk4"]:
    nav, err = [], []
    for n in [8, 16, 32, 64, 128, 256, 512, 1024, 2048]:
        r = passo_fixo(f_cos, (0, 8), 1.0, 8 / n, metodo)
        nav.append(r.avaliacoes)
        err.append(abs(r.y[-1, 0] - exato8))
    fig.add_trace(go.Scatter(x=nav, y=err, mode="lines+markers", name=nomes[metodo], line=dict(color=cores[metodo])))
nav, err = [], []
for rtol in 10.0 ** -np.arange(2, 12):
    r = rk45(f_cos, (0, 8), [1.0], rtol=rtol, atol=1e-14)
    nav.append(r.avaliacoes)
    err.append(abs(r.y[-1, 0] - exato8))
fig.add_trace(go.Scatter(x=nav, y=err, mode="lines+markers", name="RK45 adaptativo (rtol de 1e-2 a 1e-11)",
                         line=dict(color=CORES["destaque"])))
fig.update_xaxes(type="log", title="avaliações de f")
fig.update_yaxes(type="log", title="erro em t = 8", exponentformat="power")
fig.update_layout(width=800, height=470, title="Erro em função do custo, y' = y cos t", legend=dict(x=0.6, y=0.98))
mostrar(fig)

**O que notar:** em escala log-log, cada método de passo fixo é uma reta de inclinação $-p$. Para um erro de $10^{-6}$, Euler precisaria de dezenas de milhões de avaliações, Heun de alguns milhares e o RK4 de algumas centenas. Para erros abaixo de $10^{-4}$, o RK45 adaptativo fica na frente de todos, porque é de ordem 5 e põe os passos onde eles fazem falta. Repare também que o erro do RK45 **não é** igual a `rtol`: ele acompanha a tolerância, mas pode ficar acima ou abaixo dela por um fator grande.

### 5.2 Onde o RK45 põe os passos

Arraste o slider para mudar a tolerância do RK45 (com `atol = rtol` × 1 km) numa volta da órbita da seção 1.

In [ ]:
def quadro_passos(rtol):
    r = rk45(orbita, (0, T_orb), u0, rtol=rtol, atol=rtol * np.array([1, 1, 1e-3, 1e-3]))
    fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.12,
                        subplot_titles=["passos sobre a órbita", "tamanho do passo ao longo da volta"])
    th = np.linspace(0, 2 * np.pi, 200)
    fig.add_trace(go.Scatter(x=R_T * np.cos(th) / 1e3, y=R_T * np.sin(th) / 1e3, fill="toself", mode="lines",
                             line=dict(color=CORES["referencia"]), fillcolor="rgba(153,153,153,0.3)",
                             showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=r.y[:, 0] / 1e3, y=r.y[:, 1] / 1e3, mode="markers", name="fim de cada passo",
                             marker=dict(color=CORES["aproximacao"], size=6)), 1, 1)
    fig.add_trace(go.Scatter(x=r.t[1:] / 3600, y=np.diff(r.t), mode="lines+markers", name="passo h",
                             line=dict(color=CORES["aproximacao"]), marker=dict(size=5)), 1, 2)
    fig.update_xaxes(title="x (mil km)", range=[-50, 12], row=1, col=1)
    fig.update_yaxes(title="y (mil km)", range=[-26, 26], scaleanchor="x", row=1, col=1)
    fig.update_xaxes(title="t (h)", row=1, col=2)
    fig.update_yaxes(title="passo (s)", type="log", range=[0, 4.3], row=1, col=2)
    fig.update_layout(width=980, height=460, showlegend=False, margin=dict(t=110),
                      title=f"rtol = {rtol:.0e}: {len(r.t) - 1} passos, {r.rejeitados} rejeitados, "
                            f"{r.avaliacoes} avaliações".replace("e-0", "e-"))
    return fig

quadros = [(f"{t_:.0e}".replace("e-0", "e-"), quadro_passos(t_)) for t_ in [1e-3, 1e-6, 1e-9]]
mostrar(com_controle(quadros, rotulo="rtol = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** os passos ficam densos no perigeu (à direita na órbita, no início e no fim da volta) e esparsos no apogeu. Fora do início, em que a rotina ainda procura o passo certo, o passo varia por um fator de dezenas ao longo de uma volta. Cada fator 1000 na tolerância multiplica o número de passos por 2,5 a 3,5, perto do fator $1000^{1/5} \approx 4$ previsto pela ordem 5.

### 5.3 Uma EDO de ordem 2: o pêndulo de grande amplitude

Para pequenas oscilações, $\sin\theta \approx \theta$ e o período é $T_0 = 2\pi\sqrt{L/g}$, independente da amplitude. Para amplitudes grandes, isso deixa de valer. Arraste o slider para mudar a amplitude inicial $\theta_0$ de um pêndulo com $L = 1$ m. O período é medido com um **evento**: o instante em que $\omega$ volta a zero.

In [ ]:
g_L = 9.81
T0_pend = 2 * np.pi / np.sqrt(g_L)

def pendulo(t, u):
    return [u[1], -g_L * np.sin(u[0])]

def volta(t, u):            # ω = 0 com ω crescendo: fim de meia oscilação, partindo do repouso em θ0 > 0
    return u[1]
volta.direction = 1

def quadro_pendulo(graus):
    th0 = np.radians(graus)
    s = solve_ivp(pendulo, (0, 4 * T0_pend), [th0, 0.0], rtol=1e-10, atol=1e-12, dense_output=True, events=volta)
    T_num = 2 * s.t_events[0][0]                    # o evento acontece em meia oscilação
    T_ex = 4 * ellipk(np.sin(th0 / 2) ** 2) / np.sqrt(g_L)
    tt = np.linspace(0, 4 * T0_pend, 600)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tt, y=np.degrees(th0 * np.cos(np.sqrt(g_L) * tt)), mode="lines", name="linear (sen θ ≈ θ)",
                             line=dict(color=CORES["referencia"], dash="dash")))
    fig.add_trace(go.Scatter(x=tt, y=np.degrees(s.sol(tt)[0]), mode="lines", name="não linear (solve_ivp)",
                             line=dict(color=CORES["aproximacao"])))
    fig.update_xaxes(title="t (s)")
    fig.update_yaxes(title="θ (graus)", range=[-185, 185])
    fig.update_layout(width=820, height=430, legend=dict(orientation="h", y=-0.22),
                      title=f"θ₀ = {graus}°: período {numero(T_num, 4)} s (exato {numero(T_ex, 4)} s; linear "
                            f"{numero(T0_pend, 4)} s)")
    return fig

quadros = [(f"{g_}°", quadro_pendulo(g_)) for g_ in [10, 45, 90, 150, 175]]
mostrar(com_controle(quadros, rotulo="θ₀ = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** até uns 20°, as duas curvas se sobrepõem. A 90°, o período já é 18 % maior que o linear, e a 175° é quase o triplo, porque o pêndulo demora muito perto do topo. O período medido pelo evento concorda com a fórmula exata (uma integral elíptica, que é um problema do capítulo 11) em todos os algarismos mostrados. Verificar uma EDO por uma **quantidade derivada**, como o período, é tão útil quanto comparar a curva inteira.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Integre a órbita do satélite (modelo de dois corpos, perigeu a 600 km, apogeu a 39 700 km, partindo do perigeu) por 3 dias com `scipy.integrate.solve_ivp` e dê a posição final em km."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
from scipy.integrate import solve_ivp

MU = 398600.4418  # km^3/s^2

def eq_orbita(t, u):
    x, y, vx, vy = u
    r3 = (x**2 + y**2) ** 1.5
    return [vx, vy, -MU * x / r3, -MU * y / r3]

u_inicial = [r_p, 0.0, 0.0, v_p]
sol = solve_ivp(eq_orbita, (0, 3 * 86400), u_inicial)
x_f, y_f = sol.y[0, -1], sol.y[1, -1]
print(f"posição após 3 dias: x = {x_f:.0f} km, y = {y_f:.0f} km (r = {np.hypot(x_f, y_f):.0f} km)")
print(f"integração concluída: {sol.message}  ({sol.t.size - 1} passos)")
print("O solve_ivp usa passo adaptativo (RK45) com controle de erro, então o resultado é confiável.")

O código é correto como transcrição da física: o sistema de primeira ordem está certo, as unidades são coerentes e a integração terminou sem aviso. A última frase é que é o problema.

### 6.2 Verificação

**Uma lei de conservação.** No problema de dois corpos, a energia específica $\varepsilon = \tfrac12 v^2 - \mu/r$ e o momento angular $h = x v_y - y v_x$ são **constantes**. Não é preciso solução exata para checar isso:

In [ ]:
def energia(u):
    return 0.5 * (u[2] ** 2 + u[3] ** 2) - MU / np.hypot(u[0], u[1])

def momento(u):
    return u[0] * u[3] - u[1] * u[2]

eps = energia(sol.y)
hmom = momento(sol.y)
print(f"variação relativa da energia em 3 dias:         {(eps[-1] - eps[0]) / abs(eps[0]):+.2%}")
print(f"variação relativa do momento angular em 3 dias: {(hmom[-1] - hmom[0]) / hmom[0]:+.2%}")

In [ ]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.12,
                    subplot_titles=["trajetória do agente (3 dias)", "energia específica"])
dens = solve_ivp(eq_orbita, (0, 3 * 86400), u_inicial, dense_output=True)
tt = np.linspace(0, 3 * 86400, 4000)
uu = dens.sol(tt)
th = np.linspace(0, 2 * np.pi, 200)
fig.add_trace(go.Scatter(x=R_T * np.cos(th) / 1e3, y=R_T * np.sin(th) / 1e3, fill="toself", mode="lines",
                         line=dict(color=CORES["referencia"]), fillcolor="rgba(153,153,153,0.3)", showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=uu[0] / 1e3, y=uu[1] / 1e3, mode="lines", line=dict(color=CORES["erro"], width=1.5),
                         name="agente (rtol padrão)"), 1, 1)
fig.add_trace(go.Scatter(x=ref.sol(np.linspace(0, T_orb, 400))[0] / 1e3, y=ref.sol(np.linspace(0, T_orb, 400))[1] / 1e3,
                         mode="lines", line=dict(color=CORES["exata"], dash="dash"), name="órbita verdadeira"), 1, 1)
fig.add_trace(go.Scatter(x=sol.t / 3600, y=eps, mode="lines+markers", line=dict(color=CORES["erro"]),
                         marker=dict(size=5), name="energia (agente)", showlegend=False), 1, 2)
fig.add_hline(y=eps[0], line=dict(color=CORES["exata"], dash="dash"), row=1, col=2)
fig.update_xaxes(title="x (mil km)", row=1, col=1)
fig.update_yaxes(title="y (mil km)", scaleanchor="x", row=1, col=1)
fig.update_xaxes(title="t (h)", row=1, col=2)
fig.update_yaxes(title="ε (km²/s²)", row=1, col=2)
fig.update_layout(width=980, height=440, legend=dict(orientation="h", y=-0.22))
mostrar(fig)

A energia cai em degraus, um a cada passagem pelo perigeu, e a órbita encolhe e gira a cada volta. Uma órbita de dois corpos é uma elipse **fechada**: o que a figura mostra é erro numérico, não física.

**Um estudo de tolerância.** Se o resultado for confiável, apertar a tolerância não deve mudá-lo:

In [ ]:
print(" rtol      passos   avaliações    x final (km)   y final (km)")
finais = {}
for rtol in [1e-3, 1e-6, 1e-9, 1e-12]:
    s = solve_ivp(eq_orbita, (0, 3 * 86400), u_inicial, rtol=rtol, atol=rtol * 1e-3)
    finais[rtol] = s.y[:2, -1]
    print(f"{rtol:6.0e}  {s.t.size - 1:7d}  {s.nfev:10d}   {s.y[0, -1]:12.3f}   {s.y[1, -1]:12.3f}")

Da tolerância padrão para `1e-6`, a posição final muda de **dezenas de milhares de quilômetros**. De `1e-9` para `1e-12`, muda cerca de 20 metros: só aí o resultado começa a estar convergido na escala de que a antena precisa.

**Contra a solução de Kepler.** Para o modelo de dois corpos, a posição no instante $t$ sai da **equação de Kepler**, $M = E - e\sin E$, com a anomalia média $M = 2\pi t/T$. É uma equação de raiz, que o método de Newton do capítulo 3 resolve em poucas iterações:

In [ ]:
def posicao_kepler(t):
    M = 2 * np.pi * t / T_orb
    E = newton(lambda E: E - e_orb * np.sin(E) - M, M, fprime=lambda E: 1 - e_orb * np.cos(E), tol=1e-14)
    return np.array([a_orb * (np.cos(E) - e_orb), a_orb * np.sqrt(1 - e_orb**2) * np.sin(E)])

p_kepler = posicao_kepler(3 * 86400)
print(f"Kepler:                x = {p_kepler[0]:.3f} km, y = {p_kepler[1]:.3f} km")
for rtol, p in finais.items():
    print(f"erro com rtol = {rtol:5.0e}: {np.linalg.norm(p - p_kepler):10.3f} km")

O resultado do agente erra a posição por cerca de 32 mil km: a antena apontaria para o lado errado do céu.

### 6.3 Por quê

A tolerância padrão, `rtol=1e-3`, limita o erro de **cada passo** a cerca de 0,1 %. Na passagem pelo perigeu, onde a aceleração muda depressa, cada passo perde um pouco de energia. A energia determina o semieixo maior, $a = -\mu/(2\varepsilon)$, e o semieixo determina o período, $T = 2\pi\sqrt{a^3/\mu}$. Um erro de energia vira um erro de **período**, e um erro de período cresce em **fase** a cada volta:

In [ ]:
a_num = -MU / (2 * eps[-1])
T_num = 2 * np.pi * np.sqrt(a_num**3 / MU)
print(f"período da órbita numérica no fim: {T_num / 3600:.2f} h (verdadeiro: {T_orb / 3600:.2f} h)")
print(f"em 3 dias são {3 * 86400 / T_orb:.1f} voltas; um erro de {abs(T_num - T_orb) / T_orb:.1%} no período "
      f"vira até {3 * 86400 / T_orb * abs(T_num - T_orb) / T_orb:.2f} volta de atraso ou adiantamento")

Erros de fase são o modo de falha típico de simulações longas de sistemas oscilatórios: a forma da solução parece razoável, mas o "relógio" do sistema está errado. Quanto mais longa a simulação, mais apertada precisa ser a tolerância.

### 6.4 A correção

Apertar a tolerância até que o resultado pare de mudar na precisão de que se precisa, e preferir um método de ordem alta quando a tolerância é apertada:

In [ ]:
for metodo in ["RK45", "DOP853"]:
    s = solve_ivp(eq_orbita, (0, 3 * 86400), u_inicial, method=metodo, rtol=1e-10, atol=1e-10)
    erro_km = np.linalg.norm(s.y[:2, -1] - p_kepler)
    print(f"{metodo:6s} com rtol = 1e-10: erro = {erro_km * 1000:8.1f} m, {s.nfev} avaliações, "
          f"energia varia {(energia(s.y[:, -1]) - eps[0]) / abs(eps[0]):+.1e}")

Com `rtol=1e-10`, o erro cai para metros, e o `DOP853` chega lá com menos avaliações que o `RK45`. A simulação inteira leva uma fração de segundo.

> **Lição.** `rtol` e `atol` controlam o erro **local**, por passo, e os valores padrão de `solve_ivp` (`1e-3` e `1e-6`) servem para desenhar uma curva, não para fazer uma previsão. **Diga a tolerância no pedido**, e verifique o resultado com um **estudo de tolerância** e, quando o problema tiver, com uma **quantidade conservada** (energia, massa, momento). Escolha `atol` na escala de cada componente: aqui, quilômetros para a posição e metros por segundo para a velocidade.

## 7. Quando falha

### 7.1 A solução explode em tempo finito

Algumas EDOs têm soluções que vão a infinito num instante finito. Um exemplo simples é $y' = 1 + y^2$, $y(0) = 0$, cuja solução é $\tan t$, infinita em $t = \pi/2$. Uma rotina adaptativa reduz o passo sem parar perto da explosão:

In [ ]:
s = solve_ivp(lambda t, y: 1 + y**2, (0, 2), [0.0])
print(f"status = {s.status}: {s.message}")
print(f"último instante alcançado: t = {s.t[-1]:.6f} (π/2 = {np.pi / 2:.6f}), y = {s.y[0, -1]:.3e}")
print(f"últimos passos: {np.diff(s.t)[-4:]}")

O `solve_ivp` para e avisa, com `status = -1`, mas devolve o que calculou até ali. Um código que só lê `sol.y[:, -1]` trata esse ponto como o resultado em $t = 2$. **Sempre confira `sol.status`.** Explosões em tempo finito aparecem em modelos de ignição, de crescimento populacional com cooperação e de fratura; a explosão pode ser física (o modelo deixa de valer) ou um erro no modelo.

### 7.2 Lados direitos descontínuos

Uma válvula que abre num instante, um atrito que muda de sentido, um controlador liga-desliga: o lado direito tem um **salto**. As fórmulas de erro supõem $f$ suave, e a ordem cai. Para $y' = -y + u(t)$, com $u$ passando de 0 a 1 em $t = 1/3$:

In [ ]:
u_degrau = lambda t: 1.0 if t >= 1 / 3 else 0.0
f_salto = lambda t, y: -y + u_degrau(t)
exato_salto = 1 - np.exp(-(1 - 1 / 3))                      # y(1), com y(0) = 0
hs = 1 / np.array([10, 20, 40, 80, 160])
erros = [abs(passo_fixo(f_salto, (0, 1), 0.0, h, "rk4").y[-1, 0] - exato_salto) for h in hs]
print(f"RK4 com o salto entre os nós: ordem observada = {ordem_observada(hs, erros):.2f}")
s1 = solve_ivp(f_salto, (0, 1), [0.0])                       # tolerâncias padrão
s2a = solve_ivp(f_salto, (0, 1 / 3), [0.0])
s2b = solve_ivp(f_salto, (1 / 3, 1), s2a.y[:, -1])
print(f"solve_ivp atravessando o salto: {s1.nfev} avaliações, erro = {abs(s1.y[0, -1] - exato_salto):.1e}")
print(f"solve_ivp parando no salto:     {s2a.nfev + s2b.nfev} avaliações, erro = {abs(s2b.y[0, -1] - exato_salto):.1e}")

O RK4 de passo fixo cai para ordem 1. Com as tolerâncias padrão, o `solve_ivp` atravessa o salto sem perceber e erra na **segunda** casa decimal, embora `rtol` seja $10^{-3}$: o estimador de erro só "vê" o salto se um estágio cair depois dele. Integrar em dois trechos, parando no instante do salto, custa o dobro de avaliações e é milhares de vezes mais preciso. Quando o instante do salto depende do estado (a válvula fecha quando o nível chega a um valor), use um **evento** com `terminal=True` para parar ali e recomeçar.

### 7.3 Um problema que o RK45 resolve devagar

Para $y' = -1000\,(y - \cos t)$, a solução segue $\cos t$ de perto, e é suave. Mesmo assim:

In [ ]:
f_rig = lambda t, y: -1000 * (y - np.cos(t))
for rtol in [1e-2, 1e-4, 1e-6]:
    s = solve_ivp(f_rig, (0, 10), [0.0], rtol=rtol)
    print(f"RK45, rtol = {rtol:.0e}: {s.nfev:6d} avaliações")
s = solve_ivp(f_rig, (0, 10), [0.0], method="BDF", rtol=1e-6)
print(f"BDF,  rtol = 1e-06: {s.nfev:6d} avaliações")

O RK45 gasta cerca de 20 mil avaliações para seguir uma curva suave, e **afrouxar a tolerância quase não ajuda**. É o sintoma de um problema **rígido**: o passo não é limitado pela precisão, mas pela estabilidade do método. É o assunto do capítulo 13.

## Resumo

| Método | Ordem | Avaliações por passo | Quando usar |
| --- | --- | --- | --- |
| Euler | 1 | 1 | ensino; nunca em produção |
| Heun (RK2) | 2 | 2 | tolerâncias frouxas, $f$ barata |
| RK4 | 4 | 4 | passo fixo, problemas suaves; dados em malha uniforme |
| RK45 (Dormand-Prince) | 5(4) | 6 | padrão para problemas não rígidos |
| DOP853 | 8(5,3) | 12 | tolerâncias apertadas, simulações longas |
| Implícitos (`Radau`, `BDF`, `LSODA`) | variável | resolve um sistema | problemas rígidos (cap. 13) |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que, para $y' = \lambda y$, um passo do RK4 multiplica $y$ por $1 + z + z^2/2 + z^3/6 + z^4/24$, com $z = h\lambda$.
2. Escreva a EDO $m\ddot x + c\dot x + kx = F(t)$ (um sistema massa-mola-amortecedor) como um sistema de primeira ordem. Quantas condições iniciais são necessárias?
3. Mostre que, se $f$ não depende de $y$, o RK4 é a regra de Simpson e Heun é a regra do trapézio.
4. Uma integração com `rtol=1e-6` dá 1000 passos. Quantos passos você espera com `rtol=1e-9` para um método de ordem 5? E para um de ordem 8?

**Verificação (com agente)**

5. Peça a um agente um RK4 de passo fixo para sistemas e um teste. Verifique a ordem com uma EDO **não autônoma** (que dependa de $t$), como $y' = -y + \sin t$. Um RK4 que avalia todos os estágios em $t_n$ passa no teste com $y' = -y$ e falha com esse.
6. Peça a um agente o alcance de um projétil com arrasto quadrático, usando `solve_ivp` com um **evento** para o instante em que ele toca o chão. Compare com a solução sem arrasto quando o coeficiente de arrasto tende a zero, e com o alcance obtido pegando "o primeiro ponto de `sol.y` abaixo do chão". Explique a diferença.
7. Peça a um agente para passar o modelo da órbita para três dimensões, com uma órbita inclinada, e acrescentar o achatamento da Terra (termo $J_2$). Faça o estudo de tolerância da seção 6 de novo e verifique que o vetor momento angular **não** é mais conservado (só a sua componente ao longo do eixo da Terra), mas a energia é. Por que essa diferença?

**Projeto aberto**

8. **Janela de visibilidade.** Uma estação em solo vê o satélite quando ele está acima de 10° de elevação. Com `events`, calcule os instantes de início e fim de cada janela de visibilidade em 3 dias, para o modelo de dois corpos (com a rotação da Terra) e com $J_2$. Quanto muda o horário das janelas no terceiro dia?

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 25 e 26.
- E. Hairer, S. P. Nørsett e G. Wanner, *Solving Ordinary Differential Equations I: Nonstiff Problems*, 2ª ed., Springer, 1993.
- L. F. Shampine, I. Gladwell e S. Thompson, *Solving ODEs with MATLAB*, Cambridge University Press, 2003 (as ideias valem para o `solve_ivp`).
- H. D. Curtis, *Orbital Mechanics for Engineering Students*, 4ª ed., Butterworth-Heinemann, 2020, capítulos 2 e 3.